# tutorial base-designing scoring and optimizing

This notebook is part of the neurodesign-plus 2.0 tutorial audit set.

It teaches the current public workflow:

- `Experiment` stores the requested specification.
- `Design` stores one realized schedule with conceptual-trial and event metadata.
- `Optimisation` searches over designs and the authoritative public selection path is `selected_design(rank)`.

Timing is separated into:

- `event_durations`
- `trial_start_interval`
- `post_event_interval`
- `event_transition_interval`
- `inter_trial_interval`
- optional boundary rests via `rest_every_n_trials` and `rest_interval`


In [1]:
from pathlib import Path
from copy import deepcopy
import json
import os
import warnings

import numpy as np

from neurodesign import Design, Experiment, Optimisation, report

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".tmp_mpl"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore", message='install "ipywidgets" for Jupyter support')
np.set_printoptions(suppress=True, precision=3)

DEFAULT_WEIGHTS = [0.0, 0.5, 0.25, 0.25]

TRIAL_TEMPLATES = [
    {
        "template_id": "standard",
        "trial_type": "standard",
        "events": [
            {"category": "cue_easy", "code": 0, "duration": 0.8},
            {
                "category": "choice_left",
                "code": 2,
                "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            },
            {"category": "feedback", "code": 6, "duration": 1.0},
        ],
    },
    {
        "template_id": "hint_branch",
        "trial_type": "hint",
        "events": [
            {"category": "cue_hard", "code": 1, "duration": 0.8},
            {"category": "hint", "code": 4, "duration": 0.7},
            {
                "category": "choice_right",
                "code": 3,
                "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            },
            {"category": "feedback", "code": 6, "duration": 1.0},
        ],
    },
    {
        "template_id": "hold_branch",
        "trial_type": "hold",
        "events": [
            {"category": "cue_easy", "code": 0, "duration": 0.8},
            {"category": "hold", "code": 5, "duration": 0.9},
            {
                "category": "choice_left",
                "code": 2,
                "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            },
            {"category": "feedback", "code": 6, "duration": 1.0},
        ],
    },
]

EVENT_TRANSITION_INTERVAL = {
    "by_event_transition": {
        ("cue_easy", "choice_left"): {"model": "uniform", "min": 0.3, "max": 1.0},
        ("cue_hard", "hint"): {"model": "uniform", "min": 0.3, "max": 0.9},
        ("hint", "choice_right"): {"model": "uniform", "min": 0.2, "max": 0.8},
        ("cue_easy", "hold"): {"model": "uniform", "min": 0.2, "max": 0.8},
        ("hold", "choice_left"): {"model": "uniform", "min": 0.3, "max": 0.9},
        ("choice_left", "feedback"): {"model": "uniform", "min": 0.2, "max": 0.7},
        ("choice_right", "feedback"): {"model": "uniform", "min": 0.2, "max": 0.7},
    }
}

INTER_TRIAL_INTERVAL = {"model": "uniform", "min": 1.0, "mean": 1.45, "max": 1.9}

COMMON_SPEC = {
    "TR": 1.0,
    "P": [0.18, 0.12, 0.16, 0.12, 0.10, 0.10, 0.22],
    "C": [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1],
        [0, 0, 1, -1, 0, 0, 0],
        [1, -1, 0, 0, 0, 0, 0],
    ],
    "rho": 0.3,
    "n_stimuli": 7,
    "resolution": 0.1,
    "trial_templates": TRIAL_TEMPLATES,
    "trial_start_interval": 0.0,
    "post_event_interval": 0.0,
    "event_transition_interval": EVENT_TRANSITION_INTERVAL,
    "inter_trial_interval": INTER_TRIAL_INTERVAL,
    "rest_interval": 0.0,
    "event_durations": {
        "by_event_category": {
            "cue_easy": 0.8,
            "cue_hard": 0.8,
            "choice_left": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            "choice_right": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            "hint": 0.7,
            "hold": 0.9,
            "feedback": 1.0,
        }
    },
    "trial_max": 2.2,
}


def score_design(design, weights=DEFAULT_WEIGHTS):
    design.designmatrix().FCalc(weights=weights)
    return design


def build_case10_experiment(
    *,
    seed: int = 12,
    trials: list[dict[str, str]] | None = None,
    trial_template_probabilities: list[float] | None = None,
    n_conceptual_trials: int | None = None,
):
    spec = deepcopy(COMMON_SPEC)
    spec["seed"] = seed
    if trials is not None:
        spec["trials"] = trials
    else:
        spec["trial_template_probabilities"] = (
            [0.4, 0.35, 0.25] if trial_template_probabilities is None else trial_template_probabilities
        )
        spec["n_conceptual_trials"] = 10 if n_conceptual_trials is None else n_conceptual_trials
    return Experiment(**spec)

## Case 4. Building, scoring, and stopping
Score a fixed design directly, then run two real optimization passes to inspect patience-based early stopping and the disabled-stopping path.

`convergence=k` means `k` consecutive completed generations without strict improvement in the generation-best objective score. Equality counts as no improvement. The current implementation has no minimum-delta tolerance. Early stopping does not prove a global optimum.

In [2]:
trial_templates = [
    {
        "template_id": "standard",
        "trial_type": "standard",
        "events": [
            {"category": "cue_easy", "code": 0, "duration": 0.8},
            {"category": "choice_left", "code": 1, "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2}},
            {"category": "feedback", "code": 2, "duration": 1.0},
        ],
    },
    {
        "template_id": "hint_branch",
        "trial_type": "hint",
        "events": [
            {"category": "cue_hard", "code": 3, "duration": 0.8},
            {"category": "hint", "code": 4, "duration": 0.7},
            {"category": "choice_right", "code": 5, "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2}},
            {"category": "feedback", "code": 2, "duration": 1.0},
        ],
    },
]
fixed_exp = Experiment(
    TR=1.0,
    P=[1 / 6] * 6,
    C=[[1, 0, 0, 0, 0, -1]],
    rho=0.3,
    n_stimuli=6,
    trial_templates=trial_templates,
    trials=[{"template_id": "standard"}, {"template_id": "hint_branch"}],
    event_durations=1.0,
    trial_start_interval={"by_trial_type": {"standard": 0.5, "hint": 0.7}},
    post_event_interval={"by_event_category": {"feedback": 0.3, "default": 0.1}},
    event_transition_interval={
        "by_event_transition": {
            ("cue_easy", "choice_left"): 0.4,
            ("choice_left", "feedback"): 0.6,
            ("cue_hard", "hint"): 0.2,
            ("hint", "choice_right"): 0.3,
            ("choice_right", "feedback"): 0.5,
        }
    },
    inter_trial_interval=1.5,
    rest_every_n_trials=2,
    rest_interval=3.0,
    resolution=0.1,
    seed=5,
)
fixed_design = score_design(fixed_exp.create_design(seed=5))
{
    "mode": fixed_exp.mode,
    "n_conceptual_trials": fixed_exp.n_conceptual_trials,
    "n_events": len(fixed_design.order),
    "trial_template_ids": fixed_design.trial_template_ids,
    "trial_type_ids": fixed_design.trial_type_ids,
    "schedule_head": fixed_design.export_schedule()[:3],
}

C:\Users\vguigon\Desktop\Research_directory\Lab_SLD\neurodesign-plus\neurodesign\classes.py:822: UserWarning: the resolution is adjusted to be a multiple of the TR. New resolution: 0.1
  warnings.warn(


{'mode': 'fixed_trials',
 'n_conceptual_trials': 2,
 'n_events': 7,
 'trial_template_ids': ['standard', 'hint_branch'],
 'trial_type_ids': ['standard', 'hint'],
 'schedule_head': [{'run_event_index': 0,
   'trial_id': 0,
   'trial_index': 0,
   'trial_template_id': 'standard',
   'trial_type_id': 'standard',
   'event_index_within_trial': 0,
   'event_category': 'cue_easy',
   'event_code': 0,
   'trial_start': 0.0,
   'realized_trial_start_interval': 0.5,
   'event_onset': 0.5,
   'realized_event_duration': 0.8,
   'event_offset': 1.3,
   'realized_post_event_interval': 0.1,
   'following_event_transition_interval': 0.4,
   'following_inter_trial_interval': None,
   'following_rest_interval': None,
   'trial_end': None,
   'event_duration_rule_id': 'event_durations[cue_easy]',
   'trial_start_rule_id': 'trial_start_interval[standard]',
   'post_event_rule_id': 'post_event_interval[default]',
   'event_transition_rule_id': 'event_transition_interval[cue_easy->choice_left]',
   'inter_t

In [3]:
convergence_exp = Experiment(
    TR=2.0,
    n_trials=4,
    P=[0.5, 0.5],
    C=[[1, -1]],
    rho=0.3,
    n_stimuli=2,
    order=[0, 1, 0, 1],
    event_durations=1.0,
    trial_start_interval=0.0,
    post_event_interval=0.0,
    inter_trial_interval=0.0,
    resolution=0.1,
    seed=7,
)
patience_demo = Optimisation(
    experiment=convergence_exp,
    weights=[0.0, 0.0, 0.25, 0.25],
    preruncycles=1,
    cycles=4,
    seed=123,
    optimisation="simulation",
    G=2,
    I=1,
    outdes=1,
    convergence=1,
)
patience_demo.optimise()
patience_design = patience_demo.selected_design(0)

disabled_demo = Optimisation(
    experiment=convergence_exp,
    weights=[0.0, 0.0, 0.25, 0.25],
    preruncycles=1,
    cycles=4,
    seed=123,
    optimisation="simulation",
    G=2,
    I=1,
    outdes=1,
    convergence=None,
)
disabled_demo.optimise()
disabled_design = disabled_demo.selected_design(0)

{
    "patience_demo": {
        "generations_completed": patience_demo.generations_completed,
        "stop_reason": patience_demo.stop_reason,
        "best_score_history": [float(value) for value in patience_demo.optima],
        "selected_design_score": float(patience_design.F),
    },
    "disabled_demo": {
        "generations_completed": disabled_demo.generations_completed,
        "stop_reason": disabled_demo.stop_reason,
        "best_score_history": [float(value) for value in disabled_demo.optima],
        "selected_design_score": float(disabled_design.F),
    },
}

{'patience_demo': {'generations_completed': 2,
  'stop_reason': 'no improvement in generation-best score for 1 consecutive generation(s)',
  'best_score_history': [0.3125, 0.3125],
  'selected_design_score': 0.3125},
 'disabled_demo': {'generations_completed': 4,
  'stop_reason': None,
  'best_score_history': [0.3125, 0.3125, 0.3125, 0.3125],
  'selected_design_score': 0.3125}}

In [4]:
case10_opt = Optimisation(
    experiment=build_case10_experiment(seed=12),
    weights=DEFAULT_WEIGHTS,
    preruncycles=1,
    cycles=2,
    seed=101,
    optimisation="simulation",
    G=3,
    I=1,
    outdes=2,
    convergence=1,
    folder=Path("output") / "tutorial_report",
)
case10_opt.optimise()
selected_design = case10_opt.selected_design(0)
{
    "completed_generations": case10_opt.generations_completed,
    "stop_reason": case10_opt.stop_reason,
    "best_score_history": [float(value) for value in case10_opt.optima],
    "selected_rank_0_templates": selected_design.trial_template_ids,
    "selected_rank_0_metrics": {"F": selected_design.F, "Fd": selected_design.Fd, "Ff": selected_design.Ff, "Fc": selected_design.Fc},
    "available_selected_ranks": len(case10_opt.out),
}

C:\Users\vguigon\Desktop\Research_directory\Lab_SLD\neurodesign-plus\neurodesign\classes.py:822: UserWarning: the resolution is adjusted to be a multiple of the TR. New resolution: 0.1
  warnings.warn(


{'completed_generations': 2,
 'stop_reason': None,
 'best_score_history': [0.7456523819852432, 0.8417877856439969],
 'selected_rank_0_templates': ['hold_branch',
  'hint_branch',
  'hint_branch',
  'hold_branch',
  'standard',
  'hint_branch',
  'hint_branch',
  'hint_branch',
  'hold_branch',
  'hint_branch'],
 'selected_rank_0_metrics': {'F': 0.8417877856439969,
  'Fd': 1.1416310707608162,
  'Ff': 0.8245014245014245,
  'Fc': 0.2593875765529309},
 'available_selected_ranks': 2}